# 🧠 AI Logic Engine: Knowledge Ingestion (Firestore)

### ⚠️ အသုံးပြုပုံ အဆင့်ဆင့် (Instructions):
1. **Runtime Type:** `T4 GPU` ဖြစ်နေရပါမယ်။ (Runtime > Change runtime type > T4 GPU)
2. **Step 2:** Firebase Console မှ ဒေါင်းလုဒ်လုပ်ထားသော `serviceAccountKey.json` ကို Upload တင်ပါ။
3. **Step 5:** `Run All` လုပ်ထားလျှင်ပင် Step 5 ရောက်သည့်အခါ **Target Goal** (ဘယ်နှစ်ခု သွင်းမလဲ) ကို Manual ရိုက်ထည့်ပေးရပါမည်။

---

In [ ]:
# Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Complete.")

In [ ]:
# Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Memory Connected to Firestore: {DATABASE_ID}")

In [ ]:
# Step 3: Core Logic Utilities
STATE_FILE = "state_v3.json"

def clean_id(text):
    if not text: return ""
    text = text.lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

def normalize_verb(v):
    is_a_patterns = ['is_a', 'is a', 'is type of', 'belongs to', 'category of']
    if any(p == v.strip().lower() for p in is_a_patterns): return 'is_a'
    return v.strip().lower().replace(' ', '_')

def save_progress(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count}, f)

def load_progress():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Utilities Armed.")

In [ ]:
# Step 4: Load AI Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print("⏳ Loading Model (This may take a few minutes)... ")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model.eval()
print("✅ AI Logic Model is ready.")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
try:
    TARGET_INPUT = input("Enter Target Goal (Number of facts to ingest, e.g., 5000): ")
    TARGET_GOAL = int(TARGET_INPUT) if TARGET_INPUT else 1000
except:
    TARGET_GOAL = 1000
    print("Invalid input. Defaulting to 1000 facts.")

CATEGORIES = ["Biology", "Nuclear Physics", "Ancient History", "Neuroscience", "Economics", "Space Exploration"]
current_total = load_progress()
pbar = tqdm(initial=current_total, total=TARGET_GOAL, desc="Ingesting Symbols")

while current_total < TARGET_GOAL:
    try:
        # Choose Category
        cat = CATEGORIES[current_total % len(CATEGORIES)]
        prompt = f"Extract 12 unique symbolic fact triplets about {cat}. Format: S|V|O|ExplainableSentence. No intro text."
        messages = [
            {"role": "system", "content": "Output ONLY pipe-separated data: Subject|Verb|Object|Sentence. No numbering, no chat."},
            {"role": "user", "content": prompt}
        ]
        
        # Generate
        input_txt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_txt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            out = model.generate(inputs.input_ids, max_new_tokens=512, do_sample=True, temperature=0.7)
        
        response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Firestore Batch Write
        batch = db.batch()
        added_count = 0
        
        for line in response.strip().split('\n'):
            parts = [x.strip() for x in line.split('|')]
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    # Logic Mapping
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    if v == 'is_a':
                        data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        data['relations'] = firestore.ArrayUnion([{
                            'verb': v, 'targetId': oid, 
                            'sentence': parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}.",
                            'weight': 100
                        }])
                    batch.set(node_ref, data, merge=True)
                    added_count += 1
        
        if added_count > 0:
            batch.commit()
            current_total += added_count
            pbar.update(added_count)
            save_progress(current_total)
            
        # RAM Maintenance
        if current_total % 50 == 0: torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"⚠️ Loop Error: {e}. Cooling down 5s...")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Stopping ingestion. State saved.")
        break

pbar.close()
print(f"✅ Knowledge Ingestion Done. Current Total Facts in DB: {current_total}")